In [ ]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")
hf_token=os.getenv("HF_TOKEN")

from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi

from langchain_huggingface import HuggingFaceEndpoint
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline


from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

import gradio as gr
import time



reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")



In [ ]:
folder_path="./Data"

loader = DirectoryLoader(
    folder_path,
    glob="**/*",


)


docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

- used clear_func to formate the text for better embeding

In [ ]:

import re
def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isinstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)

In [ ]:
# from langchain_chroma import Chroma

# vector_store = Chroma.from_documents(
#     documents=cleaned_docs,
#     collection_name="Lang_Rag_hf",
#     embedding=embd,
#     persist_directory="./Rag/chroma_DB",
# )

# retriever = vector_store.as_retriever(
#     search_type="mmr",
#     search_kwargs={"k":7}
# )



In [ ]:
pipline= pipeline(
    "text-generation",
    model="microsoft/phi-2",
    max_new_tokens=512,
    temperature=0.3
)

In [ ]:
llm = HuggingFacePipeline(pipeline=pipline) 
template = """Given the following context, answer the question accurately.
Context: {context}
Question: {question}
Answer:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])
qa_chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
 

 

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,  
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
    length_function=len,
)
cleaned_chunk = text_split.split_documents(cleaned_docs)


vector_store = Chroma.from_documents(
    documents=cleaned_chunk,
    collection_name="Lang_Rag_hf",
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7}
)
 

text_chunks = [chunk.page_content for chunk in cleaned_chunk]
tokenized_chunks = [chunk.split() for chunk in text_chunks]
bm25 = BM25Okapi(tokenized_chunks)


 

def retrieve(query, top_k=5):
    

    vector_docs = retriever.invoke(query)
    vector_texts = [doc.page_content for doc in vector_docs]
    
    tokenized_query = query.split()
    bm25_texts = bm25.get_top_n(tokenized_query, text_chunks, n=top_k)
    
    all_texts = []
    seen = set()
    for text in bm25_texts + vector_texts:
        if text not in seen:
            seen.add(text)
            all_texts.append(text)
    
    return all_texts[:top_k*2]

def rerank(query, texts):
    if 'reranker' not in globals():
        return texts[:3]
    
    pairs = [[query, text] for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, texts), reverse=True)
    return [text for score, text in ranked[:3]]

def ask(question):
    
    texts = retrieve(question)
    
    top_texts = rerank(question, texts)
    
    if 'qa_chain' in globals():
        context = "\n\n".join(top_texts)
        response = qa_chain.run(context=context, question=question)
        if "Answer:" in response:
            response = response.split("Answer:")[-1].strip()
        return response
    else:
        return f"Found {len(top_texts)} relevant passages."

# print(ask("Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"))




print("Testing ask function:")
test_result = ask("test")
print(f"ask test result: {type(test_result)}")

def gradio_chat(message, history):
    """Simple wrapper for ask function"""
    try:
        
        result = ask(message)
        
        
        if not isinstance(result, str):
            result = str(result)
        
        return result
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(error_msg)
        return error_msg

#
demo = gr.ChatInterface(
    fn=gradio_chat,
    title="RAG Document Chatbot",
    description="Ask questions",
    examples=[
        "Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"
    ]
)



demo.launch(share=False, server_port=7861, debug=True)